# Task 2: Classification of ECG Beats Based on the Holdout Splitting Method

In this notebook, we will:
1. Load the resampled training and testing data from CSV files
2. Prepare the data for classification
3. Train a Support Vector Machine (SVM) classifier
4. Evaluate the model using accuracy, precision, recall, F1-score, and confusion matrix
5. Visualize the results

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix,
    classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 8)

## 2. Load the Data

Load the resampled training and testing data that were created in the previous notebook (data_split_resample.ipynb).

In [ ]:
# Load the resampled beat holdout data
train_data = np.loadtxt('train_beats.csv', delimiter=',')
test_data = np.loadtxt('test_beats.csv', delimiter=',')

print(f"Training data shape: {train_data.shape}")
print(f"Testing data shape: {test_data.shape}")

## 3. Prepare the Data

The data structure is as follows:
- Columns 0-274: ECG signal features (275 time points)
- Column 275 (index -2): Class label (1-8)
- Column 276 (index -1): Patient number

We need to:
1. Separate features (X) from labels (y)
2. Remove the patient number column (not needed for beat holdout)
3. Standardize the features for better SVM performance

In [ ]:
# Define class names for visualization
class_names = {
    0: 'Undetermined',
    1: 'Normal (N)',
    2: 'LBBBB (L)',
    3: 'RBBBB (R)',
    4: 'PVC (V)',
    5: 'APB (A)',
    6: 'Fusion VN (F)',
    7: 'Fusion PN (f)',
    8: 'Paced (/)'
}

# Extract features and labels
X_train = train_data[:, :-2]  # All columns except last two (class and patient)
y_train = train_data[:, -2].astype(int)  # Second to last column is class label

X_test = test_data[:, :-2]
y_test = test_data[:, -2].astype(int)

print(f"\nFeature shape:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"\nLabel shape:")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

# Check class distribution
print("\nTraining set class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  Class {cls} ({class_names[cls]}): {count} samples")

print("\nTesting set class distribution:")
unique, counts = np.unique(y_test, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  Class {cls} ({class_names[cls]}): {count} samples")

## 4. Feature Standardization

SVM performance is sensitive to feature scaling. We'll use StandardScaler to normalize the features to have zero mean and unit variance.

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature standardization complete.")
print(f"Training features - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")
print(f"Testing features - Mean: {X_test_scaled.mean():.6f}, Std: {X_test_scaled.std():.6f}")

## 5. Train Support Vector Machine (SVM) Classifier

We'll use an SVM with RBF (Radial Basis Function) kernel, which is effective for non-linear classification problems like ECG beat classification.

**SVM Parameters:**
- `C`: Regularization parameter (controls trade-off between smooth decision boundary and classifying training points correctly)
- `gamma`: Kernel coefficient for RBF (controls influence of individual training samples)
- `kernel`: 'rbf' for Radial Basis Function kernel

In [ ]:
# Initialize and train SVM classifier
print("Training SVM classifier...")
print("This may take several minutes depending on your dataset size...\n")

# Using RBF kernel with default parameters
# You can tune C and gamma for better performance
svm_classifier = SVC(
    kernel='rbf',      # Radial Basis Function kernel
    C=1.0,             # Regularization parameter
    gamma='scale',     # Kernel coefficient (1 / (n_features * X.var()))
    random_state=42,
    verbose=True       # Show training progress
)

# Train the model
svm_classifier.fit(X_train_scaled, y_train)

print("\nTraining complete!")

## 6. Make Predictions

In [ ]:
# Make predictions on test set
print("Making predictions on test set...")
y_pred = svm_classifier.predict(X_test_scaled)
print("Predictions complete!")

## 7. Model Evaluation Function

Create a comprehensive evaluation function that computes and displays all required metrics.

In [ ]:
def evaluate_model(y_true, y_pred, class_names_dict, model_name="Model"):
    """
    Comprehensive model evaluation function.
    
    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    class_names_dict : dict
        Dictionary mapping class IDs to class names
    model_name : str
        Name of the model for display
    
    Returns:
    --------
    metrics : dict
        Dictionary containing all computed metrics
    """
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    precision_weighted = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    recall_weighted = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    # Per-class metrics
    precision_per_class = precision_score(y_true, y_pred, average=None, zero_division=0)
    recall_per_class = recall_score(y_true, y_pred, average=None, zero_division=0)
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # Print overall metrics
    print(f"\n{'='*60}")
    print(f"{model_name} - Evaluation Metrics")
    print(f"{'='*60}\n")
    
    print(f"Overall Metrics:")
    print(f"  Accuracy:           {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"\n  Macro Average:")
    print(f"    Precision:        {precision_macro:.4f}")
    print(f"    Recall:           {recall_macro:.4f}")
    print(f"    F1-Score:         {f1_macro:.4f}")
    print(f"\n  Weighted Average:")
    print(f"    Precision:        {precision_weighted:.4f}")
    print(f"    Recall:           {recall_weighted:.4f}")
    print(f"    F1-Score:         {f1_weighted:.4f}")
    
    # Print per-class metrics
    print(f"\n{'='*60}")
    print(f"Per-Class Metrics:")
    print(f"{'='*60}\n")
    
    # Get unique classes present in the data
    unique_classes = np.unique(np.concatenate([y_true, y_pred]))
    
    print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support'}")
    print(f"{'-'*70}")
    
    for i, cls in enumerate(unique_classes):
        class_name = class_names_dict.get(cls, f"Class {cls}")
        support = np.sum(y_true == cls)
        print(f"{class_name:<20} {precision_per_class[i]:<12.4f} {recall_per_class[i]:<12.4f} {f1_per_class[i]:<12.4f} {support}")
    
    # Store metrics in dictionary
    metrics = {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'precision_weighted': precision_weighted,
        'recall_macro': recall_macro,
        'recall_weighted': recall_weighted,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'confusion_matrix': cm
    }
    
    return metrics

## 8. Evaluate the Model

In [ ]:
# Evaluate the SVM model
metrics = evaluate_model(y_test, y_pred, class_names, model_name="SVM (Beat Holdout)")

## 9. Visualize Confusion Matrix

The confusion matrix shows how many samples from each true class were predicted as each class.

In [ ]:
def plot_confusion_matrix(cm, class_names_dict, title='Confusion Matrix', figsize=(12, 10)):
    """
    Plot confusion matrix as a heatmap.
    
    Parameters:
    -----------
    cm : array-like
        Confusion matrix
    class_names_dict : dict
        Dictionary mapping class IDs to class names
    title : str
        Title for the plot
    figsize : tuple
        Figure size
    """
    plt.figure(figsize=figsize)
    
    # Get class labels from the confusion matrix
    unique_classes = np.arange(cm.shape[0])
    class_labels = [class_names_dict.get(i, f"Class {i}") for i in unique_classes]
    
    # Create heatmap
    sns.heatmap(
        cm, 
        annot=True,           # Show numbers in cells
        fmt='d',              # Format as integers
        cmap='Blues',         # Color scheme
        xticklabels=class_labels,
        yticklabels=class_labels,
        cbar_kws={'label': 'Count'},
        square=True,
        linewidths=0.5,
        linecolor='gray'
    )
    
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Plot confusion matrix
plot_confusion_matrix(
    metrics['confusion_matrix'], 
    class_names, 
    title='SVM Confusion Matrix (Beat Holdout Method)'
)

## 10. Visualize Per-Class Performance

Create bar plots to compare precision, recall, and F1-score across different classes.

In [ ]:
def plot_per_class_metrics(metrics, class_names_dict, y_test, figsize=(14, 6)):
    """
    Plot per-class precision, recall, and F1-score.
    
    Parameters:
    -----------
    metrics : dict
        Dictionary containing evaluation metrics
    class_names_dict : dict
        Dictionary mapping class IDs to class names
    y_test : array-like
        True labels for test set
    figsize : tuple
        Figure size
    """
    # Get unique classes present in test data
    unique_classes = np.unique(y_test)
    class_labels = [class_names_dict.get(i, f"Class {i}") for i in unique_classes]
    
    # Extract per-class metrics
    precision = metrics['precision_per_class']
    recall = metrics['recall_per_class']
    f1 = metrics['f1_per_class']
    
    # Create bar plot
    fig, ax = plt.subplots(figsize=figsize)
    
    x = np.arange(len(class_labels))
    width = 0.25
    
    bars1 = ax.bar(x - width, precision, width, label='Precision', color='#3498db', alpha=0.8)
    bars2 = ax.bar(x, recall, width, label='Recall', color='#2ecc71', alpha=0.8)
    bars3 = ax.bar(x + width, f1, width, label='F1-Score', color='#e74c3c', alpha=0.8)
    
    # Add value labels on bars
    def add_value_labels(bars):
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}',
                   ha='center', va='bottom', fontsize=9)
    
    add_value_labels(bars1)
    add_value_labels(bars2)
    add_value_labels(bars3)
    
    ax.set_xlabel('Class', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('Per-Class Performance Metrics (SVM - Beat Holdout)', 
                fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(class_labels, rotation=45, ha='right')
    ax.legend(loc='lower right', fontsize=10)
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.show()

# Plot per-class metrics
plot_per_class_metrics(metrics, class_names, y_test)

## 11. Classification Report

Generate a detailed classification report using sklearn.

In [ ]:
# Get unique classes from test set
unique_classes = np.unique(y_test)
target_names = [class_names[i] for i in unique_classes]

print("\nDetailed Classification Report:")
print("="*80)
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

## 12. Summary and Key Observations

### Summary:
- **Method**: Beat Holdout Validation (75% train, 25% test)
- **Classifier**: Support Vector Machine (SVM) with RBF kernel
- **Data**: Resampled training data with balanced classes

### Key Points:
1. **Class Balance**: The training data was resampled to have equal representation across all classes, which helps the model learn all classes equally.

2. **Feature Standardization**: Standardizing features is crucial for SVM performance as it's sensitive to feature scales.

3. **Beat Holdout Limitation**: This method randomly splits beats from all patients, which can lead to data leakage (beats from same patient in both train and test sets). This is why patient holdout validation (Task 3) is also important.

### Expected Performance:
- High accuracy expected due to beat holdout method (potential data leakage)
- Performance comparison with patient holdout method will reveal generalization capability

### Next Steps:
- **Task 3**: Implement patient holdout validation
- **Task 4**: Apply explainability techniques (permutation feature importance)
- **Task 5**: Compare different classifiers
- **Task 6**: Optional clustering analysis

## 13. Save Model Results (Optional)

In [ ]:
# Optionally save the trained model
import pickle

# Save model
with open('svm_beat_holdout_model.pkl', 'wb') as f:
    pickle.dump(svm_classifier, f)
    
# Save scaler
with open('scaler_beat_holdout.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save metrics
with open('metrics_beat_holdout.pkl', 'wb') as f:
    pickle.dump(metrics, f)

print("Model, scaler, and metrics saved successfully!")